[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/69_bilinear_resize_solution.ipynb)

# 🟡 Solution: Bilinear Resize NCHW

Reference solution for `bilinear_resize`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F


In [ ]:
# ✅ SOLUTION

def _resize_positions(in_size: int, out_size: int, align_corners: bool, device, dtype):
    if align_corners:
        if out_size == 1:
            return torch.zeros(out_size, device=device, dtype=dtype)
        return torch.linspace(0, in_size - 1, out_size, device=device, dtype=dtype)
    pos = (torch.arange(out_size, device=device, dtype=dtype) + 0.5) * in_size / out_size - 0.5
    return pos.clamp(0, in_size - 1)


def bilinear_resize(x: torch.Tensor, out_h: int, out_w: int,
                    align_corners: bool = False) -> torch.Tensor:
    B, C, H, W = x.shape
    ys = _resize_positions(H, out_h, align_corners, x.device, x.dtype)
    xs = _resize_positions(W, out_w, align_corners, x.device, x.dtype)
    y0 = torch.floor(ys).long()
    x0 = torch.floor(xs).long()
    y1 = (y0 + 1).clamp(max=H - 1)
    x1 = (x0 + 1).clamp(max=W - 1)
    wy = (ys - y0.to(x.dtype)).view(1, 1, out_h, 1)
    wx = (xs - x0.to(x.dtype)).view(1, 1, 1, out_w)

    Ia = x[:, :, y0][:, :, :, x0]
    Ib = x[:, :, y0][:, :, :, x1]
    Ic = x[:, :, y1][:, :, :, x0]
    Id = x[:, :, y1][:, :, :, x1]
    top = Ia * (1 - wx) + Ib * wx
    bottom = Ic * (1 - wx) + Id * wx
    return top * (1 - wy) + bottom * wy


In [ ]:
# Verify
x = torch.randn(2, 3, 5, 7)
y = bilinear_resize(x, 9, 4)
print(y.shape)
print((y - F.interpolate(x, size=(9, 4), mode='bilinear', align_corners=False)).abs().max())


In [ ]:
# Run judge
from torch_judge import check
check('bilinear_resize')
